# Les types (`dtype`) et les valeurs manquantes

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_accidents

df = load_accidents()
df.dtypes

## 1. Les dtypes classiques

| dtype pandas | Équivalent Python / NumPy | Exemple |
|---|---|---|
| `int64` | `int` | `42` |
| `float64` | `float` | `3.14` |
| `object` | `str` (ou mélange) | `"Paris"` |
| `bool` | `bool` | `True` |
| `datetime64[ns]` | `datetime` | `2024-01-15` |
| `timedelta64[ns]` | `timedelta` | `3 jours` |
| `category` | — | `Categorical` |

> **Piège classique** : une colonne d'entiers qui contient un `NaN` devient `float64`.
> C'est le problème historique numéro un de pandas. On y revient en §5.

In [ ]:
# Démonstration du piège : entier + NaN → float
s_int = pd.Series([1, 2, 3])
print("Sans NaN :", s_int.dtype)

s_nan = pd.Series([1, 2, None])  # None force la conversion
print("Avec NaN :", s_nan.dtype)

## 2. Les dtypes modernes (pandas 2.0+) — les types nullables

Depuis pandas 1.0, et généralisé en 2.0, il existe des types **nullables** :
`Int64`, `Float64`, `string`, `boolean` (avec majuscule).

Ces types utilisent `pd.NA` au lieu de `np.nan` et **ne corrompent pas le dtype** en présence de manquants.

In [ ]:
# Ancien monde : int + NaN → float (type corrompu)
s_old = pd.Series([1, 2, None], dtype="float64")
print("dtype ancien :", s_old.dtype, "→", s_old.tolist())

# Nouveau monde : Int64 nullable (I majuscule) — dtype préservé
s_new = pd.Series([1, 2, None], dtype="Int64")
print("dtype nouveau:", s_new.dtype, "→", s_new.tolist())

In [ ]:
# Les 4 types nullables principaux
exemples = pd.DataFrame({
    "entier":   pd.array([1, 2, None], dtype="Int64"),
    "flottant": pd.array([1.5, None, 3.0], dtype="Float64"),
    "texte":    pd.array(["a", None, "c"], dtype="string"),
    "booleen":  pd.array([True, None, False], dtype="boolean"),
})
print(exemples.dtypes)
exemples

In [ ]:
# pd.NA vs np.nan : le manquant universel des types nullables
print(type(pd.NA))      # <NA>
print(type(np.nan))     # float

# np.nan est un float — il corrompt les colonnes d'entiers
# pd.NA est typé — il reste NA dans n'importe quel dtype nullable

## 3. ArrowDtype — le backing columnar moderne

In [ ]:
# Depuis pandas 1.5, les dtypes peuvent être "backed" par Apache Arrow
# Arrow est le format in-memory standard de l'écosystème data (parquet, duckdb, polars)
# Avantages : mémoire réduite, compatibilité native avec parquet/duckdb/polars

try:
    import pyarrow
    df_arrow = df.convert_dtypes(dtype_backend="pyarrow")
    print("Dtypes avec backend Arrow :")
    print(df_arrow.dtypes.head(8))
    print(f"\nMémoire numpy : {df.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
    print(f"Mémoire arrow : {df_arrow.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
except ImportError:
    print("pyarrow non installé : uv add pyarrow")

## 4. `.convert_dtypes()` — laisser pandas choisir

`.convert_dtypes()` infère les meilleurs types modernes pour chaque colonne.

In [ ]:
df_converti = df.convert_dtypes()

# Comparaison avant/après
comparison = pd.DataFrame({
    "avant": df.dtypes,
    "après": df_converti.dtypes,
})
comparison[comparison["avant"] != comparison["après"]]

## 5. Les valeurs manquantes — section majeure

Les valeurs manquantes sont à la fois **le sujet le plus douloureux** et **le plus mal géré** en pratique.
Cette section est longue parce que c'est 30-40 % de la douleur réelle en pandas.

### 5.1 Pourquoi c'est compliqué — l'histoire

Pandas a utilisé `np.nan` (un flottant) pour représenter les manquants dès le début.
Problème : `np.nan` est un `float` — forcer un entier à cohabiter avec `np.nan` force
la colonne entière à devenir `float64`.

Depuis pandas 1.0 : les dtypes nullables (`Int64`, `Float64`, `string`, `boolean`) utilisent `pd.NA`,
une valeur manquante typée qui ne corrompt pas le dtype.

### 5.2 Les quatre représentations à reconnaître

| Représentation | Dtype concerné | Quand on le voit |
|---|---|---|
| `np.nan` | `float64`, `object` | Colonnes numériques, legacy |
| `None` | `object` | Colonnes de strings/mixed |
| `pd.NaT` | `datetime64` | Colonnes de dates |
| `pd.NA` | `Int64`, `Float64`, `string`, `boolean` | Types nullables modernes |

In [ ]:
# Observer les différentes représentations
s_float  = pd.Series([1.0, np.nan, 3.0])
s_object = pd.Series(["a", None, "c"])
s_date   = pd.Series(pd.to_datetime(["2024-01-01", None, "2024-03-01"]))
s_Int64  = pd.Series([1, None, 3], dtype="Int64")

for name, s in [("float64", s_float), ("object", s_object), ("datetime64", s_date), ("Int64", s_Int64)]:
    print(f"{name:12s} → valeur manquante : {s[1]!r}  dtype: {s.dtype}")

### 5.3 Détecter les manquants

In [ ]:
# Réflexe numéro 2 après .info() : regarder les manquants
df.isna().mean().sort_values(ascending=False).round(3)

In [ ]:
# Nombre absolu + pourcentage par colonne
manquants = pd.DataFrame({
    "n_manquants": df.isna().sum(),
    "pct":         (df.isna().mean() * 100).round(1),
}).query("n_manquants > 0").sort_values("pct", ascending=False)
manquants

**Avant d'exécuter la cellule suivante — que va retourner `df[df["lat"] == np.nan]` ?**

In [ ]:
# NaN n'est égal à rien, pas même à lui-même
print("np.nan == np.nan :", np.nan == np.nan)  # False !

# Conséquence : cette comparaison ne trouve jamais rien
print("df[df['lat'] == np.nan] :", df[df["lat"] == np.nan].shape)

# La bonne façon
print("df[df['lat'].isna()]    :", df[df["lat"].isna()].shape)

### 5.4 Supprimer les manquants — `.dropna()`

In [ ]:
# .dropna() par défaut : supprime TOUTES les lignes avec au moins un NaN
# Extrêmement destructeur sur un vrai dataset
print(f"Avant dropna() total   : {len(df)} lignes")
print(f"Après dropna() total   : {len(df.dropna())} lignes  ({len(df.dropna())/len(df)*100:.1f}% conservé)")

In [ ]:
# subset : ne supprimer que si les colonnes importantes sont manquantes
cols_critiques = ["dep", "mois", "lum"]
apres_subset = df.dropna(subset=cols_critiques)
print(f"Après dropna(subset={cols_critiques}) : {len(apres_subset)} lignes  ({len(apres_subset)/len(df)*100:.1f}%)")

In [ ]:
# how='all' : supprimer seulement si TOUTES les colonnes du subset sont NaN
df.dropna(subset=cols_critiques, how="all").shape

### 5.5 Remplir les manquants — `.fillna()`

In [ ]:
# Remplissage par une constante
df["lat"].fillna("0").isna().sum()

In [ ]:
# Propagation forward (ffill) — valeur précédente
# Utile pour les séries temporelles avec des trous
s = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan])
print("ffill :", s.ffill().tolist())
print("bfill :", s.bfill().tolist())

In [ ]:
# Remplir par la médiane du groupe — plus prudent que la moyenne globale
df["vma"].fillna(df["vma"].median())

> **Avertissement** : remplir par une moyenne ou une médiane **biaise silencieusement** toutes les analyses ultérieures.
> Ce n'est pas un choix neutre — c'est une décision métier qui doit être documentée et justifiée.
> Demandez-vous toujours : *pourquoi cette valeur est-elle manquante ?* Avant de choisir comment la traiter.

### 5.6 Les manquants dans `groupby` et dans les jointures

In [ ]:
# groupby exclut silencieusement les lignes où la clé est NaN
# (sauf avec dropna=False — vu dans le notebook 4.0)
from _data import load_penguins
penguins = load_penguins()

print("NaN dans sex :", penguins["sex"].isna().sum())
print("Total pingouins :", len(penguins))
print("Comptés par groupby(sex) :", penguins.groupby("sex")["species"].count().sum())

In [ ]:
# Dans un merge, les NaN ne joignent pas entre eux
df1 = pd.DataFrame({"k": [1, np.nan, 3], "a": ["A", "B", "C"]})
df2 = pd.DataFrame({"k": [1, np.nan, 4], "b": ["X", "Y", "Z"]})

# NaN dans df1 et NaN dans df2 ne se joignent pas (même avec how='inner' ou 'outer')
df1.merge(df2, on="k", how="outer")

### 5.7 Règle pratique

In [ ]:
# À appeler systématiquement juste après .info() sur un nouveau dataset
df.isna().mean().sort_values(ascending=False)

> **Réflexe** : `df.isna().mean().sort_values(ascending=False)` — toujours,
> sur tout nouveau dataset, avant toute analyse.

## 6. Changer de type : `.astype()` et ses pièges

In [ ]:
# Conversion simple
df["mois"].astype("Int8")  # économie mémoire : int64 → Int8 (1 octet au lieu de 8)

In [ ]:
# Piège 1 : cast object → int qui explose sur un NaN
s = pd.Series(["1", "2", None, "4"])
try:
    s.astype(int)
except Exception as e:
    print(type(e).__name__, ":", e)

# Solution : utiliser Int64 nullable ou gérer les NaN d'abord
s.astype("Int64")

In [ ]:
# Piège 2 : cast float → str produit '1.0' au lieu de '1'
s_float = pd.Series([1.0, 2.0, 3.0])
print("astype(str)       :", s_float.astype(str).tolist())  # ['1.0', '2.0', '3.0']

# Solution : passer par int d'abord
print("astype(int).str() :", s_float.astype(int).astype(str).tolist())  # ['1', '2', '3']

In [ ]:
# pd.to_numeric() avec errors='coerce' — convertit ce qui peut l'être, NaN sinon
s_mixte = pd.Series(["1", "deux", "3", "quatre", "5"])
pd.to_numeric(s_mixte, errors="coerce")

## Récapitulatif

| Besoin | Outil |
|---|---|
| Voir les types | `df.dtypes` / `df.info()` |
| Convertir vers les types modernes | `df.convert_dtypes()` |
| Type entier qui tolère les NaN | `Int64` (I majuscule) |
| Type string moderne | `string` |
| Vue d'ensemble des manquants | `df.isna().mean().sort_values(ascending=False)` |
| Supprimer les lignes manquantes | `df.dropna(subset=["col_critique"])` |
| Remplir les manquants | `df.fillna(valeur)` / `.ffill()` / `.bfill()` |
| Changer de type | `.astype("dtype")` |
| Conversion tolérante aux erreurs | `pd.to_numeric(errors="coerce")` |